# Database Administration

## Introduction

The previous notebook covered reading from an existing database. This notebook covers how to *create* and *modify* one — the fundamental CRUD operations (Create, Read, Update, Delete) that every database user needs to know.

## Objectives

You will be able to:

- Name the four SQLite data types and choose the right one for a given column
- Create a table with `CREATE TABLE` and define its schema
- Insert rows with `INSERT INTO`
- Modify existing rows with `UPDATE`
- Remove rows with `DELETE`
- Commit changes with `conn.commit()` and explain why it matters

---

## SQLite Data Types

SQLite uses four storage classes. Declaring the right type enforces data integrity and makes aggregation functions (SUM, AVG) work correctly.

| Type | Use for | Examples |
|------|---------|----------|
| `INTEGER` | Whole numbers | age, count, id, year |
| `REAL` | Decimal numbers | price, weight, score (up to 15 significant digits) |
| `TEXT` | Strings | name, city, email |
| `BLOB` | Raw binary data | images, files stored as bytes |

SQLite is lenient — if you write `INT`, `VARCHAR`, or `FLOAT`, it will map them to the closest category above. But `INTEGER`, `REAL`, `TEXT`, and `BLOB` are the canonical names.

---

## Creating a Database

Connecting to a path that doesn't exist creates the database file automatically. We'll use `':memory:'` here so every cell can be re-run without leftover state.

In [ ]:
import sqlite3
import pandas as pd

# ':memory:' creates a temporary in-memory database — great for demos and notebooks
conn = sqlite3.connect(':memory:')
cur = conn.cursor()

---

## CREATE TABLE

Define the table name and each column's name and data type. The `PRIMARY KEY` column is auto-incremented — you don't insert values for it.

```sql
CREATE TABLE cats (
    id    INTEGER PRIMARY KEY,
    name  TEXT,
    age   INTEGER,
    breed TEXT
);
```

In [ ]:
cur.execute("""
    CREATE TABLE cats (
        id    INTEGER PRIMARY KEY,
        name  TEXT,
        age   INTEGER,
        breed TEXT
    );
""")

---

## INSERT INTO

Add rows with `INSERT INTO`. List the column names you want to fill, then the matching values. Since `id` is an auto-incrementing primary key, omit it — SQLite assigns the value.

```sql
INSERT INTO cats (name, age, breed)
VALUES ('Maru', 3, 'Scottish Fold');
```

In [ ]:
cur.execute("INSERT INTO cats (name, age, breed) VALUES ('Maru', 3, 'Scottish Fold');")
cur.execute("INSERT INTO cats (name, age, breed) VALUES ('Hana', 1, 'Tabby');")
cur.execute("INSERT INTO cats (name, age, breed) VALUES ('Lil Bub', 5, 'American Shorthair');")
cur.execute("INSERT INTO cats (name, age, breed) VALUES ('Moe', 10, 'Tabby');")
cur.execute("INSERT INTO cats (name, age, breed) VALUES ('Patches', 2, 'Calico');")

# Verify
cur.execute("SELECT * FROM cats;").fetchall()

---

## ALTER TABLE

Add a column to an existing table with `ALTER TABLE`. You cannot change a column's type or remove a column in SQLite — only add.

```sql
ALTER TABLE cats ADD COLUMN owner_name TEXT;
```

In [ ]:
cur.execute("ALTER TABLE cats ADD COLUMN owner_name TEXT;")

cur.execute("SELECT * FROM cats;").fetchall()

New rows default to `NULL` in the added column until you update them.

---

## UPDATE

`UPDATE` changes values in existing rows. Always use a `WHERE` clause — without one, every row in the table is updated.

```sql
UPDATE cats
SET owner_name = 'Alice'
WHERE name = 'Maru';
```

In [ ]:
cur.execute("UPDATE cats SET owner_name = 'Alice' WHERE name = 'Maru';")
cur.execute("UPDATE cats SET owner_name = 'Alice' WHERE name = 'Hana';")
cur.execute("UPDATE cats SET owner_name = 'Bob' WHERE name = 'Moe';")

cur.execute("SELECT * FROM cats;")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## DELETE

`DELETE` removes rows that match a `WHERE` condition. Like `UPDATE`, omitting `WHERE` deletes everything.

```sql
DELETE FROM cats WHERE name = 'Lil Bub';
```

Using the `PRIMARY KEY` is safer than matching on name — names might not be unique.

In [ ]:
# Remove Lil Bub by primary key
cur.execute("DELETE FROM cats WHERE id = 3;")

cur.execute("SELECT * FROM cats;")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## Committing Changes

Changes made through a connection are only visible to *that* connection until you call `conn.commit()`. Other connections to the same database file see a snapshot from before your un-committed changes.

In [ ]:
# Write to a real file to demonstrate commit behaviour
conn_file = sqlite3.connect('data/database_admin_101/pets_database.db')
cur_file = conn_file.cursor()

# Create the table and insert a row (not yet committed)
try:
    cur_file.execute("""
        CREATE TABLE cats (
            id INTEGER PRIMARY KEY,
            name TEXT,
            age INTEGER,
            breed TEXT
        );
    """)
except Exception:
    pass  # table already exists on re-run

cur_file.execute("INSERT INTO cats (name, age, breed) VALUES ('Maru', 3, 'Scottish Fold');")

# Second connection opened before commit — sees no data
conn2 = sqlite3.connect('data/database_admin_101/pets_database.db')
cur2 = conn2.cursor()
print("Before commit:", cur2.execute("SELECT * FROM cats;").fetchall())

In [ ]:
conn_file.commit()

# Reconnect to see the committed data
conn2 = sqlite3.connect('data/database_admin_101/pets_database.db')
cur2 = conn2.cursor()
print("After commit: ", cur2.execute("SELECT * FROM cats;").fetchall())

---

## Practice: Build a School Database

Design and populate a database for a school. Use `':memory:'` so you can re-run freely.

In [ ]:
school_conn = sqlite3.connect(':memory:')
school_cur = school_conn.cursor()

In [ ]:
# Create a 'students' table with columns: id (PK), first_name, last_name, grade_level, gpa
# Choose appropriate data types for each column


In [ ]:
# Insert at least 5 students into the table


In [ ]:
# Query the table to verify your inserts — wrap in a DataFrame


In [ ]:
# One student's GPA changed — update it


In [ ]:
# Create a second table 'courses' with columns: id (PK), course_name, teacher, credits (INTEGER)


In [ ]:
# Insert at least 3 courses


In [ ]:
# Delete a student who dropped out


In [ ]:
# Commit your changes
school_conn.commit()

---

## Summary

In this notebook you learned how to:

- Choose between SQLite's four data types: `INTEGER`, `REAL`, `TEXT`, `BLOB`
- Create a table with `CREATE TABLE` and an auto-incrementing `PRIMARY KEY`
- Add rows with `INSERT INTO`, update them with `UPDATE ... SET ... WHERE`, and remove them with `DELETE ... WHERE`
- Extend a table schema with `ALTER TABLE ... ADD COLUMN`
- Persist changes to disk with `conn.commit()`

Next: [03 — Aggregation and Groupby](03_aggregation_and_groupby.ipynb)